# 생성자-평가자와 정지 조건

생성자와 평가자를 분리하고 세 조건 가운데 하나가 충족되면 반복을 멈춥니다.

## 실행 환경과 비용 상한

최대 3회 반복, 회당 생성·평가 2회, 마지막 보고 1회 이하입니다. 계획 비용은 0.0217달러 미만입니다.

## 코드를 읽기 전에

반복 한 차례는 생성 호출과 평가 호출로 구성됩니다. 생성자는 과제·이전 피드백을 받아 `answer`를 만들고, 평가자는 같은 기준으로 `score`와 피드백을 만듭니다. 프로그램은 점수가 숫자이며 범위 안인지 검사한 뒤 `history`에 반복 번호, 점수, 직전 점수 대비 개선 폭을 한 행으로 저장합니다. 초안은 `draft`, 다음 반복에 쓸 피드백은 `feedback` 변수에 따로 유지합니다. 실제 모델 호출은 각 함수에서 `responses.create()`가 실행될 때 발생합니다.

정지 순서는 결과 해석의 일부입니다. 85점 이상이면 통과, 두 번째 반복부터 개선 폭이 3점 미만이면 정체, 어느 조건도 아니면 최대 세 번까지 실행합니다. 최대 반복으로 끝났다는 사실은 품질 기준을 통과했다는 뜻이 아닙니다.

In [1]:
# 이 장은 OpenAI SDK를 직접 배우므로 두 패키지만 고정합니다.
%pip install -q openai==2.29.0 python-dotenv==1.2.3


Note: you may need to restart the kernel to use updated packages.


### 다음 셀을 읽는 방법

`load_dotenv(find_dotenv(usecwd=True))`는 현재 폴더에서 위로 올라가며 `.env`를 찾아 환경 변수에 넣습니다. `OpenAI(timeout=45.0, max_retries=0)`는 동기 클라이언트를 만들며 아직 모델을 호출하지 않습니다. `calls`는 이후 성공 응답의 지연과 토큰 수를 모을 빈 목록입니다.


In [2]:
# 동기 API 호출과 사용량 측정에 필요한 도구를 준비합니다.
import json
import os
import time
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "프로젝트 루트 .env에 OPENAI_API_KEY가 필요합니다."
MODEL = "gpt-5.4-mini"
client = OpenAI(timeout=45.0, max_retries=0)
calls: list[dict[str, float | int | str]] = []
print("모델:", MODEL, "| API 키 존재: 예")


모델: gpt-5.4-mini | API 키 존재: 예


### 다음 셀을 읽는 방법

`call_json(instructions, prompt, limit)`은 지시문·사용자 입력·최대 출력 토큰을 받습니다. 함수 안의 `client.responses.create()`에서 실제 요청이 발생하며, 응답의 JSON 텍스트를 파이썬 사전으로 바꾸어 반환합니다. `calls`에는 성공 응답의 지연과 입력·출력 토큰만 추가됩니다.


In [3]:
# 동기 Responses API의 JSON 응답과 사용량을 기록합니다.
def call_json(instructions: str, prompt: str, limit: int = 240) -> dict:
    started = time.perf_counter()
    response = client.responses.create(
        model=MODEL,
        instructions=instructions + " 반드시 유효한 JSON 객체만 반환하세요.",
        input="JSON 요청\n" + prompt,
        max_output_tokens=limit,
        reasoning={"effort": "none"},
        text={"format": {"type": "json_object"}},
    )
    elapsed = time.perf_counter() - started
    usage = response.usage
    calls.append({"latency_s": elapsed, "input": usage.input_tokens,
                  "output": usage.output_tokens})
    return json.loads(response.output_text)


### 다음 셀을 읽는 방법

`call_answer()`는 `answer` 문자열 하나만 허용하는 JSON Schema를 요청에 넣습니다. SDK 응답을 사전으로 바꾼 뒤 값이 문자열이며 비어 있지 않은지 다시 검사하고 사전을 반환합니다. 스키마 통과는 답의 사실 정확성까지 보장하지 않습니다.


In [4]:
# 동기 응답의 answer 형식을 엄격하게 검사합니다.
def call_answer(instructions: str, prompt: str, limit: int = 400) -> dict:
    started = time.perf_counter()
    response = client.responses.create(
        model=MODEL, instructions=instructions, input=prompt,
        max_output_tokens=limit, reasoning={"effort": "none"},
        text={"format": {"type": "json_schema", "name": "answer_payload",
              "schema": {"type": "object", "properties": {"answer": {"type": "string"}},
              "required": ["answer"], "additionalProperties": False}, "strict": True}},
    )
    calls.append({"latency_s": time.perf_counter() - started,
                  "input": response.usage.input_tokens, "output": response.usage.output_tokens})
    data = json.loads(response.output_text)
    if not isinstance(data.get("answer"), str) or not data["answer"].strip():
        raise ValueError("응답의 answer가 비어 있습니다.")
    return data


### 다음 셀을 읽는 방법

`cost_usd(records)`는 앞에서 모은 성공 응답의 입력·출력 토큰을 합산해 이 실습에 고정한 단가를 적용합니다. `score_of(data)`는 평가 점수가 불리언이 아닌 숫자이며 0~100 범위인지 확인한 뒤 정수로 반환합니다. 두 함수 모두 모델을 호출하지 않습니다.


In [5]:
# 반환된 토큰 사용량으로 할인 없는 비용을 추정합니다.
def cost_usd(records: list[dict]) -> float:
    inputs = sum(int(row["input"]) for row in records)
    outputs = sum(int(row["output"]) for row in records)
    return inputs * 0.75 / 1_000_000 + outputs * 4.50 / 1_000_000
def score_of(data: dict) -> int:
    value = data.get("score")
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise ValueError("평가 응답의 score는 숫자여야 합니다.")
    if not 0 <= value <= 100:
        raise ValueError("평가 응답의 score는 0~100이어야 합니다.")
    return int(value)


## 품질 기준과 반복 상한

In [6]:
# 생성자와 평가자 반복에 적용할 세 정지 조건을 선언합니다.
TASK = "2단계 인증 도입 안내를 120~180자로 작성하세요. 2026년 10월 15일, 전 직원, help@company.test를 포함하고 등록 절차를 1)·2)·3)으로 표시합니다."
MAX_ITERATIONS = 3
QUALITY_THRESHOLD = 85
MIN_IMPROVEMENT = 3
print({"최대 반복": MAX_ITERATIONS, "품질 임계": QUALITY_THRESHOLD,
       "최소 개선 폭": MIN_IMPROVEMENT})


{'최대 반복': 3, '품질 임계': 85, '최소 개선 폭': 3}


## 생성과 평가를 번갈아 실행

In [7]:
# 초안 생성과 평가를 반복하고 실제 점수에 따라 중단 여부를 판단합니다.
history: list[dict] = []
draft = ""
feedback = "첫 초안을 작성하세요."
previous_score: int | None = None
stop_reason = "max_iterations"
for iteration in range(1, MAX_ITERATIONS + 1):
    generated = call_answer("사내 안내문 생성자입니다.",
                          f"과제: {TASK}\n이전 피드백: {feedback}\nJSON answer", 260)
    draft = generated["answer"]
    judged = call_json("독립 평가자입니다. 요구 충족도를 0~100으로 채점합니다.",
                       f"과제: {TASK}\n초안: {draft}\nJSON score, feedback", 180)
    score = score_of(judged)
    improvement = None if previous_score is None else score - previous_score
    history.append({"iteration": iteration, "score": score, "improvement": improvement})
    feedback = str(judged.get("feedback", ""))
    if score >= QUALITY_THRESHOLD:
        stop_reason = "quality_threshold"; break
    if improvement is not None and improvement < MIN_IMPROVEMENT:
        stop_reason = "min_improvement"; break
    previous_score = score


## 실험 결과와 중단 이유

In [8]:
# 반복 이력, 중단 이유, 최종 초안과 성공 응답 사용량을 출력합니다.
print("반복 기록:", history)
print("중단 이유:", stop_reason)
print("최종 초안:", draft)
print("성공 응답 수:", len(calls), "| 사용량 기준 추정 비용($):", round(cost_usd(calls), 6))
assert len(history) <= MAX_ITERATIONS
assert stop_reason in {"quality_threshold", "min_improvement", "max_iterations"}


반복 기록: [{'iteration': 1, 'score': 100, 'improvement': None}]
중단 이유: quality_threshold
최종 초안: 2026년 10월 15일부터 전 직원 대상 2단계 인증이 도입됩니다. 문의는 help@company.test로 연락 바랍니다. 등록 절차: 1) 계정 로그인 2) 인증 앱 설치 3) 보안코드 등록을 완료하세요.
성공 응답 수: 2 | 사용량 기준 추정 비용($): 0.000916


## 실험 결과 해석

품질 임계값에 도달하면 남은 반복을 사용하지 않습니다. 점수가 오르더라도 개선 폭이 3점보다 작으면 추가 비용의 효과가 작다고 보고 중단합니다. 두 조건에 걸리지 않으면 최대 3회에서 끝납니다. 평가 점수는 이 과제와 이 평가 프롬프트 안에서만 의미가 있으며 사람 검토를 대신하지 않습니다.

## 다른 상황에 적용하기

SQL 생성에서는 평가자가 문법 검사와 읽기 전용 규칙을 확인하고 실패 피드백을 줄 수 있습니다. 그러나 평가자가 틀린 기준을 사용하면 반복할수록 원하는 결과에서 멀어질 수 있습니다. 생성자-평가자 패턴은 명시적 기준과 개선 가능성이 있을 때 유용하며, 한 번에 검증 가능한 형식 오류에는 코드 게이트가 더 예측 가능합니다.